# Plotting the ACC specific genes from SGACC a molecular profile update and potential implications paper.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/bulk_counts.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.head()

In [ ]:
adata.obs['ClinicalOutcome'] = adata.obs['ClinicalOutcome'].map({'5Plus':'Good', '2Minus':'Poor'})
adata.obs['ModelOutcome2'] = adata.obs['ModelOutcome2'].map({'Average':'Good', 'Poor':'Poor'})
adata.obs['ModelOutcome1'] = adata.obs['ModelOutcome1'].map({'Average':'Good', 'Poor':'Poor'})

In [ ]:
adata.obs.rename(columns={
    'ClinicalOutcome':'Clinical Outcome',
    'DEGOutcome':'DEG Outcome',
    'ModelOutcome1':'ModelOutcome1',
    'ModelOutcome2':'Model Outcome'
}, inplace = True)

In [ ]:
# normalization and log transformation of expression values.
adata_tmp = adata.copy()
sc.pp.normalize_total(adata_tmp, target_sum=1e4)  # library-size normalize
sc.pp.log1p(adata_tmp)

In [ ]:
adata_tmp.var_names = adata_tmp.var['gene_label']

In [ ]:
# expression matrix: samples x genes
expr = pd.DataFrame(
    adata_tmp.X,
    index=adata_tmp.obs_names,
    columns=adata_tmp.var_names
)

In [ ]:
expr.head()

In [ ]:
# ---------------------------------------
# 2) Pick genes to show (DE-driven subset)
#    (padj<0.05, highest |log2FC|, and optionally baseMean filter)
# ---------------------------------------
de = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()

# Optional low-expression filter to avoid noisy genes
BASEMEAN_MIN = 10 # adjust as you like or set to 0 to disable
if "baseMean" in de.columns:
    de = de[de["baseMean"] >= BASEMEAN_MIN]

# choose top up/down (e.g., 25 each)
N = 20
up = de[(de["padj"] < 0.05) & (de["log2FoldChange"] > 0)].nsmallest(N, "padj")
dn = de[(de["padj"] < 0.05) & (de["log2FoldChange"] < 0)].nsmallest(N, "padj")

def names(df):
    if "gene_label" in df.columns:
        sym = df["gene_label"].astype(str)
        # Replace 'nan' with index
        return sym.where(sym != 'nan', df.index).astype(str)
    return df.index.astype(str)

genes = pd.Index(names(up).tolist() + names(dn).tolist())
genes = genes.intersection(expr.columns)  # keep only those present in expression

print(f"Found {len(up)} upregulated genes and {len(dn)} downregulated genes")
print(f"Total genes for heatmap (before adding genes of interest): {len(genes)}")

In [ ]:
# ===============================================
# OPTIMIZED HEATMAP - INPUT YOUR GENE LIST HERE
# ===============================================

# ---------------------------------------
# INPUT: Your list of genes to plot
# ---------------------------------------

input_gene_list = [
    # Tumor suppressors & cell cycle regulators
    "TP53", "CDKN2A", "CDKN2B", "ATM", "LATS1", "PLAGL1",
    "P73", "RB1", "CYLD", "TSC1", "SUFU",
    
    # Tyrosine kinase / receptor signaling
    "KIT", "C_KIT", "EGFR", "FGFR1", "FGFR2", "FGFR3", "FGFR4",
    "PDGFRA", "PDGFRB", "KDR", "RET",
    
    # PI3K–Akt–mTOR pathway
    "PIK3CA", "PTEN", "AKT1", "MTOR", "IGF1R",
    
    # NOTCH pathway & regulators
    "NOTCH2", "NOTCH3", "NOTCH4", "JAG1", "JAG2", "DLL1",
    "SPEN", "FBXW7", "RBPJ",
    
    # Chromatin remodeling and epigenetic modifiers
    "SMARCA2", "SMARCA4", "SMARCB1", "ARID1A", "ARID1B",
    "ARID4B", "ARID5B", "KDM6A", "KDM5A", "CREBBP", "EP300",
    "KMT2C", "KMT2D", "BCOR", "PBRM1", "BRD1", "BRD2",
    
    # Transcription factors & developmental regulators
    "EN1", "SOX4", "SOX11", "PRAME", "SMO", "CDK6",
    "KLF4", "FOXO1", "JUNB", "FOSB", "VGLL3",
    "ERBB3", "CTNNB1", "RUNX1", "FOXP2",
    
    # Adhesion / structural / EMT-related
    "CDH2", "CDH5", "LIMA1", "CNTN6", "HSPG2",
    "SEMA3G", "SEMA5A", "FAT3", "FAT4",
    
    # DNA damage and repair genes
    "BRCA1", "ATRX", "UHRF1", "DCLRE1A", "TXNIP",
    
    # Other recurrent or candidate genes in ACC
    "FGF16", "IL17RD", "IDH1", "APC", "NRAS", "HRAS", "BRAF",
    "LRP1B", "TERT", "SF3B1", "MLH1", "MSH6",
    "MUC12", "TFAP4", "MAP10", "OTOP", "RUNX3",
    "NBL1", "PRDM2", "NUP107", "MDM2"
]

# ---------------------------------------
# Filter genes that exist in the data
# ---------------------------------------
genes = pd.Index(input_gene_list).intersection(expr.columns)

print(f"Requested {len(input_gene_list)} genes")
print(f"Found {len(genes)} genes in expression data: {', '.join(genes)}")

if len(genes) < len(input_gene_list):
    missing = set(input_gene_list) - set(genes)
    print(f"Warning: {len(missing)} genes not found: {', '.join(missing)}")

if len(genes) == 0:
    print("ERROR: No genes found in the data!")
else:
    # ---------------------------------------
    # Build z-scored matrix
    # ---------------------------------------
    mat = expr[genes]  # samples x selected genes
    mat_z = (mat - mat.mean(axis=0)) / mat.std(axis=0)
    
    # Drop any genes with NaN or infinite values
    mat_z = mat_z.loc[:, ~mat_z.isnull().any(axis=0)]
    mat_z = mat_z.loc[:, ~np.isinf(mat_z).any(axis=0)]
    
    print(f"After removing NaN/Inf: {mat_z.shape[1]} genes remaining")
    
    # ---------------------------------------
    # Order samples by metadata (Clinical Outcome -> Site so Good/Poor form blocks)
    # ---------------------------------------
    order_cols = [c for c in ["Clinical Outcome", "Site"] if c in adata.obs.columns]
    if order_cols:
        sample_order = adata.obs.sort_values(order_cols).index
        mat_z = mat_z.loc[sample_order]
    
    # ---------------------------------------
    # Column annotations (sample bars): Clinical Outcome + Site
    # ---------------------------------------
    annotation_cols = ['Clinical Outcome', 'Site']
    col_colors_dict = {}
    
    # Curated palette for Site that deliberately avoids red/green hues so it can't
    # be confused with the Poor (#E63946) / Good (#2c9d2a) outcome colors.
    SITE_PALETTE = ['#1d3557', '#457b9d', '#8338ec', '#f4a261', '#ffb703', '#6d6875', '#adb5bd']
    
    for col in annotation_cols:
        if col in adata.obs.columns:
            unique_vals = adata.obs[col].dropna().astype(str).unique()
            
            if col == 'Clinical Outcome':
                # Fixed brand colors regardless of category order
                fixed_lut = {'Poor': '#E63946', 'Good': '#2c9d2a'}
                lut = {val: fixed_lut.get(val, '#B0B0B0') for val in unique_vals}
            elif col == 'Site':
                colors = [SITE_PALETTE[i % len(SITE_PALETTE)] for i in range(len(unique_vals))]
                lut = dict(zip(unique_vals, colors))
            else:
                palette = sns.color_palette("Set2", n_colors=len(unique_vals))
                lut = dict(zip(unique_vals, palette))
            
            col_colors_dict[col] = adata.obs.loc[mat_z.index, col].astype(str).map(lut)
    
    col_colors = pd.DataFrame(col_colors_dict) if col_colors_dict else None
    
    # ---------------------------------------
    # Identify which genes are significantly DE
    # ---------------------------------------
    de_full = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()
    
    def get_gene_labels(df):
        if "gene_label" in df.columns:
            sym = df["gene_label"].astype(str)
            return sym.where(sym != 'nan', df.index).astype(str)
        return df.index.astype(str)
    
    sig_de_genes = set(get_gene_labels(de_full[(de_full["padj"] < 0.05)]))
    non_sig_genes = [g for g in genes if g not in sig_de_genes]
    
    print(f"Non-significant genes (will be marked with *): {', '.join(non_sig_genes) if non_sig_genes else 'None'}")
    
    # ---------------------------------------
    # Plot heatmap - custom matplotlib GridSpec layout.
    #
    # Every panel gets its own fixed GridSpec cell (colorbar included) so the
    # colorbar keeps its narrow, tall shape instead of being stretched by
    # clustermap's auto-computed GridSpec. Samples stay in the outcome-ordered
    # column order; only the genes (rows) are clustered.
    # ---------------------------------------
    import os
    from scipy.cluster.hierarchy import linkage, dendrogram
    from matplotlib.patches import Patch
    from matplotlib.colors import to_rgb
    
    sample_names = mat_z.index.tolist()
    
    fig = plt.figure(figsize=(14, max(8, len(genes) * 0.28)))
    gs = fig.add_gridspec(
        nrows=3, ncols=3,
        width_ratios=[0.4, 8, 2.4],      # [colorbar column, heatmap, legend margin]
        height_ratios=[0.5, 0.6, 10],    # [colorbar row, annotation-strip row, heatmap row]
        hspace=0.04, wspace=0.04,
    )
    
    # Colorbar gets its own narrow, dedicated column so its shape stays fixed.
    ax_cbar = fig.add_subplot(gs[0, 0])
    ax_annot = fig.add_subplot(gs[1, 1])
    ax_heat = fig.add_subplot(gs[2, 1])
    
    # 1) Cluster genes for ordering only - no dendrogram is drawn.
    row_linkage = linkage(mat_z.T.values, method='average', metric='euclidean')
    dendro_data = dendrogram(row_linkage, no_plot=True)
    
    row_order = dendro_data['leaves']
    genes_ordered = mat_z.columns[row_order]
    heat_data = mat_z[genes_ordered].T.values  # genes x samples, clustered gene order
    
    # 2) Annotation strips (Clinical Outcome + Site) above the heatmap only
    annot_rgb = np.array([
        [to_rgb(c) for c in col_colors_dict[col].loc[sample_names]]
        for col in annotation_cols
    ])
    ax_annot.imshow(annot_rgb, aspect='auto', interpolation='nearest')
    ax_annot.set_yticks(range(len(annotation_cols)))
    ax_annot.set_yticklabels(annotation_cols, fontsize=8)
    ax_annot.set_xticks([])
    for spine in ax_annot.spines.values():
        spine.set_visible(False)
    
    # 3) Main heatmap
    im = ax_heat.imshow(heat_data, cmap="RdBu_r", vmin=-3, vmax=3, aspect='auto')
    ax_heat.set_xticks(range(len(sample_names)))
    ax_heat.set_xticklabels(sample_names, rotation=90, fontsize=8)
    ax_heat.set_xlabel("Samples", fontsize=10)
    ax_heat.set_ylabel("Genes (Z-score)", fontsize=10)
    
    current_labels = list(genes_ordered)
    new_labels = [f"{g} *" if g in non_sig_genes else g for g in current_labels]
    ax_heat.set_yticks(range(len(genes_ordered)))
    ax_heat.set_yticklabels(new_labels, fontsize=9)
    
    # 4) Colorbar - its own fixed, dedicated axis, so its size is predictable
    fig.colorbar(im, cax=ax_cbar, label='Z-score')
    
    # 5) Legends - one block per category, reserved legend column, can't overlap
    legend_y = 0.85
    for title in annotation_cols:
        unique_vals = adata.obs.loc[sample_names, title].astype(str).unique()
        handles = [
            Patch(
                facecolor=col_colors_dict[title][adata.obs.loc[sample_names, title].astype(str) == val].iloc[0],
                edgecolor='black', linewidth=0.5
            )
            for val in unique_vals
        ]
        leg = fig.legend(
            handles=handles,
            labels=list(unique_vals),
            title=title,
            loc='upper left',
            bbox_to_anchor=(0.83, legend_y),
            fontsize=8,
            title_fontsize=9,
            frameon=True,
        )
        fig.add_artist(leg)  # keep this legend; otherwise the next fig.legend() replaces it
        legend_y -= 0.25
    
    fig.subplots_adjust(left=0.05, right=0.98, top=0.98, bottom=0.05)
    
    # Save as SVG
    output_dir = '{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome'
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, 'heatmap_acc_genes.svg'),
                format='svg', bbox_inches='tight')
    
    plt.show()
    plt.close()

# Plotting Genes that are significantly differentially expressed in the dataset.

In [ ]:
# ===============================================
# OPTIMIZED HEATMAP - INPUT YOUR GENE LIST HERE
# ===============================================

# ---------------------------------------
# INPUT: Your list of genes to plot
# ---------------------------------------
# only significant gene list with correct ordering.
input_gene_list = [
    # Tumor suppressors & cell cycle regulators
    "TP53", "CDKN2B", "P73", "CYLD",
    
    # Tyrosine kinase / receptor signaling
    "C_KIT", "EGFR", "PDGFRA", "RET",
    
    # PI3K–Akt–mTOR pathway
    "PTEN",
    
    # NOTCH pathway & regulators
    "JAG2", "DLL1",
    
    # Chromatin remodeling and epigenetic modifiers
    "ARID5B", "KDM6A",
    
    # Transcription factors & developmental regulators
    "SOX4", "FOXO1", "ERBB3", "CTNNB1",
    
    # Adhesion / structural / EMT-related
    "CDH2", "LIMA1", "CNTN6", "HSPG2", "SEMA5A",
    
    # DNA damage and repair genes
    "BRCA1", "UHRF1", "DCLRE1A", "TXNIP",
    
    # Other recurrent or candidate genes in ACC
    "NRAS", "LRP1B", "TERT", "MAP10", "OTOP",
    "NBL1", "PRDM2", "NUP107"
]


# ---------------------------------------
# Filter genes that exist in the data
# ---------------------------------------
genes = pd.Index(input_gene_list).intersection(expr.columns)

print(f"Requested {len(input_gene_list)} genes")
print(f"Found {len(genes)} genes in expression data: {', '.join(genes)}")

if len(genes) < len(input_gene_list):
    missing = set(input_gene_list) - set(genes)
    print(f"Warning: {len(missing)} genes not found: {', '.join(missing)}")

if len(genes) == 0:
    print("ERROR: No genes found in the data!")
else:
    # ---------------------------------------
    # Build z-scored matrix
    # ---------------------------------------
    mat = expr[genes]  # samples x selected genes
    mat_z = (mat - mat.mean(axis=0)) / mat.std(axis=0)
    
    # Drop any genes with NaN or infinite values
    mat_z = mat_z.loc[:, ~mat_z.isnull().any(axis=0)]
    mat_z = mat_z.loc[:, ~np.isinf(mat_z).any(axis=0)]
    
    print(f"After removing NaN/Inf: {mat_z.shape[1]} genes remaining")
    
    # ---------------------------------------
    # Order samples by metadata (Clinical Outcome -> Site so Good/Poor form blocks)
    # ---------------------------------------
    order_cols = [c for c in ["Clinical Outcome", "Site"] if c in adata.obs.columns]
    if order_cols:
        sample_order = adata.obs.sort_values(order_cols).index
        mat_z = mat_z.loc[sample_order]
    
    # ---------------------------------------
    # Column annotations (sample bars): Clinical Outcome + Site
    # ---------------------------------------
    annotation_cols = ['Clinical Outcome', 'Site']
    col_colors_dict = {}
    
    # Curated palette for Site that deliberately avoids red/green hues so it can't
    # be confused with the Poor (#E63946) / Good (#2c9d2a) outcome colors.
    SITE_PALETTE = ['#1d3557', '#457b9d', '#8338ec', '#f4a261', '#ffb703', '#6d6875', '#adb5bd']
    
    for col in annotation_cols:
        if col in adata.obs.columns:
            unique_vals = adata.obs[col].dropna().astype(str).unique()
            
            if col == 'Clinical Outcome':
                # Fixed brand colors regardless of category order
                fixed_lut = {'Poor': '#E63946', 'Good': '#2c9d2a'}
                lut = {val: fixed_lut.get(val, '#B0B0B0') for val in unique_vals}
            elif col == 'Site':
                colors = [SITE_PALETTE[i % len(SITE_PALETTE)] for i in range(len(unique_vals))]
                lut = dict(zip(unique_vals, colors))
            else:
                palette = sns.color_palette("Set2", n_colors=len(unique_vals))
                lut = dict(zip(unique_vals, palette))
            
            col_colors_dict[col] = adata.obs.loc[mat_z.index, col].astype(str).map(lut)
    
    col_colors = pd.DataFrame(col_colors_dict) if col_colors_dict else None
    
    # ---------------------------------------
    # Identify which genes are significantly DE
    # ---------------------------------------
    de_full = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()
    
    def get_gene_labels(df):
        if "gene_label" in df.columns:
            sym = df["gene_label"].astype(str)
            return sym.where(sym != 'nan', df.index).astype(str)
        return df.index.astype(str)
    
    sig_de_genes = set(get_gene_labels(de_full[(de_full["padj"] < 0.05)]))
    non_sig_genes = [g for g in genes if g not in sig_de_genes]
    
    print(f"Non-significant genes (will be marked with *): {', '.join(non_sig_genes) if non_sig_genes else 'None'}")
    
    # ---------------------------------------
    # Plot heatmap - custom matplotlib GridSpec layout.
    #
    # Every panel gets its own fixed GridSpec cell (colorbar included) so the
    # colorbar keeps its narrow, tall shape instead of being stretched by
    # clustermap's auto-computed GridSpec. Samples stay in the outcome-ordered
    # column order; only the genes (rows) are clustered.
    # ---------------------------------------
    import os
    from scipy.cluster.hierarchy import linkage, dendrogram
    from matplotlib.patches import Patch
    from matplotlib.colors import to_rgb
    
    sample_names = mat_z.index.tolist()
    
    fig = plt.figure(figsize=(14, max(8, len(genes) * 0.28)))
    gs = fig.add_gridspec(
        nrows=3, ncols=3,
        width_ratios=[0.4, 8, 2.4],      # [colorbar column, heatmap, legend margin]
        height_ratios=[0.5, 0.6, 10],    # [colorbar row, annotation-strip row, heatmap row]
        hspace=0.04, wspace=0.04,
    )
    
    # Colorbar gets its own narrow, dedicated column so its shape stays fixed.
    ax_cbar = fig.add_subplot(gs[0, 0])
    ax_annot = fig.add_subplot(gs[1, 1])
    ax_heat = fig.add_subplot(gs[2, 1])
    
    # 1) Cluster genes for ordering only - no dendrogram is drawn.
    row_linkage = linkage(mat_z.T.values, method='average', metric='euclidean')
    dendro_data = dendrogram(row_linkage, no_plot=True)
    
    row_order = dendro_data['leaves']
    genes_ordered = mat_z.columns[row_order]
    heat_data = mat_z[genes_ordered].T.values  # genes x samples, clustered gene order
    
    # 2) Annotation strips (Clinical Outcome + Site) above the heatmap only
    annot_rgb = np.array([
        [to_rgb(c) for c in col_colors_dict[col].loc[sample_names]]
        for col in annotation_cols
    ])
    ax_annot.imshow(annot_rgb, aspect='auto', interpolation='nearest')
    ax_annot.set_yticks(range(len(annotation_cols)))
    ax_annot.set_yticklabels(annotation_cols, fontsize=8)
    ax_annot.set_xticks([])
    for spine in ax_annot.spines.values():
        spine.set_visible(False)
    
    # 3) Main heatmap
    im = ax_heat.imshow(heat_data, cmap="RdBu_r", vmin=-3, vmax=3, aspect='auto')
    ax_heat.set_xticks(range(len(sample_names)))
    ax_heat.set_xticklabels(sample_names, rotation=90, fontsize=8)
    ax_heat.set_xlabel("Samples", fontsize=10)
    ax_heat.set_ylabel("Genes (Z-score)", fontsize=10)
    
    current_labels = list(genes_ordered)
    new_labels = [f"{g} *" if g in non_sig_genes else g for g in current_labels]
    ax_heat.set_yticks(range(len(genes_ordered)))
    ax_heat.set_yticklabels(new_labels, fontsize=9)
    
    # 4) Colorbar - its own fixed, dedicated axis, so its size is predictable
    fig.colorbar(im, cax=ax_cbar, label='Z-score')
    
    # 5) Legends - one block per category, reserved legend column, can't overlap
    legend_y = 0.85
    for title in annotation_cols:
        unique_vals = adata.obs.loc[sample_names, title].astype(str).unique()
        handles = [
            Patch(
                facecolor=col_colors_dict[title][adata.obs.loc[sample_names, title].astype(str) == val].iloc[0],
                edgecolor='black', linewidth=0.5
            )
            for val in unique_vals
        ]
        leg = fig.legend(
            handles=handles,
            labels=list(unique_vals),
            title=title,
            loc='upper left',
            bbox_to_anchor=(0.83, legend_y),
            fontsize=8,
            title_fontsize=9,
            frameon=True,
        )
        fig.add_artist(leg)  # keep this legend; otherwise the next fig.legend() replaces it
        legend_y -= 0.25
    
    fig.subplots_adjust(left=0.05, right=0.98, top=0.98, bottom=0.05)
    
    # Save as SVG
    output_dir = '{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome'
    os.makedirs(output_dir, exist_ok=True)
    fig.savefig(os.path.join(output_dir, 'heatmap_acc_genes_only_significant.svg'),
                format='svg', bbox_inches='tight')
    
    plt.show()
    plt.close()

In [ ]:
# # Extract top 30 upregulated and top 30 downregulated genes
# de_genes = adata.var.dropna(subset=["padj", "log2FoldChange"]).copy()

# # Optional: apply baseMean filter
# BASEMEAN_MIN = 10
# if "baseMean" in de_genes.columns:
#     de_genes = de_genes[de_genes["baseMean"] >= BASEMEAN_MIN]

# # Get top 30 upregulated genes (positive log2FC, lowest padj)
# top30_up = de_genes[(de_genes["padj"] < 0.05) & (de_genes["log2FoldChange"] > 0)].nsmallest(30, "padj")

# # Get top 30 downregulated genes (negative log2FC, lowest padj)
# top30_down = de_genes[(de_genes["padj"] < 0.05) & (de_genes["log2FoldChange"] < 0)].nsmallest(30, "padj")

# # Display upregulated genes
# print("=" * 80)
# print("TOP 30 UPREGULATED GENES")
# print("=" * 80)
# print(f"\nFound {len(top30_up)} upregulated genes\n")
# up_display = top30_up[["Symbol", "log2FoldChange", "padj", "baseMean"]].copy()
# up_display["Symbol"] = up_display["Symbol"].astype(str)
# print(up_display.to_string())

# # Display downregulated genes
# print("\n" + "=" * 80)
# print("TOP 30 DOWNREGULATED GENES")
# print("=" * 80)
# print(f"\nFound {len(top30_down)} downregulated genes\n")
# down_display = top30_down[["Symbol", "log2FoldChange", "padj", "baseMean"]].copy()
# down_display["Symbol"] = down_display["Symbol"].astype(str)
# print(down_display.to_string())

# # Optional: Save to CSV files
# top30_up.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/top30_upregulated_genes.csv')
# top30_down.to_csv(f'{ACC_DATA_ROOT}/bulk_rna_data/top30_downregulated_genes.csv')
# print("\n" + "=" * 80)
# print("Files saved:")
# print("  - top30_upregulated_genes.csv")
# print("  - top30_downregulated_genes.csv")
# print("=" * 80)